# Phase 28 — Consolidation of revision numbers and consistency check against the manuscript
**Resolves:** R2 minor-1 (table/figure/number cross-check), supports every Study 2 edit by producing ONE file of final numbers
(`S2_revision_numbers.md`) and flagging any manuscript value that disagrees with the recomputed artefacts.
Run LAST, after Phases 16-27.

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
import glob
files = sorted(glob.glob(os.path.join(ENV["REV"], "S2_*.csv")))
print(len(files), "revision tables found")
md = ["# Study 2 revision — consolidated numbers\n"]
for f in files:
    try:
        df = pd.read_csv(f)
    except pd.errors.EmptyDataError:
        print("skipped empty table:", os.path.basename(f)); continue
    md.append(f"\n## {os.path.basename(f)}  ({len(df)} rows)\n")
    try:
        md.append(df.head(40).round(4).to_markdown(index=False))
    except Exception:                      # tabulate not installed
        md.append(df.head(40).round(4).to_string(index=False))
open(os.path.join(ENV["REV"], "S2_revision_numbers.md"), "w", encoding="utf-8").write("\n".join(md)); print("written S2_revision_numbers.md")

45 revision tables found
written S2_revision_numbers.md


In [3]:
# Manuscript values that must match the recomputed artefacts (edit if the manuscript changes)
issues = []
def chk(name, ms, val, tol=0.0015):
    ok = abs(float(ms) - float(val)) <= tol
    print(("OK   " if ok else "DIFF ") + f"{name}: manuscript {ms} vs recomputed {float(val):.4f}")
    if not ok: issues.append(name)
fin = pd.read_csv(os.path.join(ENV["TABLES"], "proposed_model_final_results.csv")).set_index("dataset")
for tag, ms in [("pakistan", 0.557), ("arab5_depression", 0.687), ("arab5_anxiety", 0.782), ("uganda", 0.697)]:
    chk(f"Table 12 AUC {tag}", ms, fin.loc[tag, "auc_roc"])
dl = pd.read_csv(os.path.join(ENV["REV"], "S2_dl_table14_complete.csv")).set_index("dataset") if os.path.exists(os.path.join(ENV["REV"], "S2_dl_table14_complete.csv")) else None
if dl is not None:
    for tag, ms in [("pakistan", 0.554)]: chk(f"Table 14 DL AUC seed42 {tag}", ms, dl.loc[tag, "dl_auc_seed42"], 0.003)
x = pd.read_csv(os.path.join(ENV["REV"], "S2_xnat_logistic_transfer.csv")) if os.path.exists(os.path.join(ENV["REV"], "S2_xnat_logistic_transfer.csv")) else None
r = pd.read_csv(os.path.join(ENV["TABLES"], "cross_country_external_validation.csv")).set_index("direction")
chk("Table 16 Pakistan->Arab external AUC", 0.480, r.loc["pakistan_to_arab5", "external_auc"]); chk("Table 16 Arab->Pakistan external AUC", 0.488, r.loc["arab5_to_pakistan", "external_auc"])
aud = pd.read_csv(os.path.join(ENV["REV"], "S2_model_selection_audit.csv"))
print("\nRF selected by the one-SE rule in:", list(aud[aud.selected_is_rf].dataset), "| NOT selected in:", list(aud[~aud.selected_is_rf].dataset))
print("\nFLAGGED:", issues if issues else "none")

OK   Table 12 AUC pakistan: manuscript 0.557 vs recomputed 0.5571
OK   Table 12 AUC arab5_depression: manuscript 0.687 vs recomputed 0.6867
OK   Table 12 AUC arab5_anxiety: manuscript 0.782 vs recomputed 0.7815
OK   Table 12 AUC uganda: manuscript 0.697 vs recomputed 0.6965
OK   Table 14 DL AUC seed42 pakistan: manuscript 0.554 vs recomputed 0.5544
OK   Table 16 Pakistan->Arab external AUC: manuscript 0.48 vs recomputed 0.4802
OK   Table 16 Arab->Pakistan external AUC: manuscript 0.488 vs recomputed 0.4876

RF selected by the one-SE rule in: ['uganda'] | NOT selected in: ['pakistan', 'arab5_depression', 'arab5_anxiety']

FLAGGED: none


### Cross-reference corrections still to apply in the Study 2 text (from the original manuscript)
| Where | Text says | Should be |
|---|---|---|
| Leakage findings, Pakistan paragraph | "(Table 10)" | Table 9 (Pakistan audit) |
| Leakage findings, Arab-region sentence | "(Table 11)" | Table 10 (Arab-region audit) |
| Table 11 (excluded variables) | not cited in the text | add a pointer, e.g. "excluded variables are listed in Table 11" |
| Model selection | "(Table 14)" | Table 13 |
| Deep learning | "(Table 15)" | Table 14 |
| Decision curves | "(Table 16)" | Table 15 |
| Cross-national | "(Table 17)" | Table 16 |
| Study 1 text (outside this revision) | "Table 3" for the correlation matrix; "Figure 8" cited with Table 5 | Table 2; check the figure pointer |
After the revision adds new tables (flow, validation strategies, selection audit, calibration, subgroup, supplementary), renumber ALL tables
and re-check every in-text pointer once more.